# Streaming Checkpoints and Fault Tolerance: Exactly-Once Recovery

**Module**: 09_streaming  
**Notebook**: 05_checkpoints_and_fault_tolerance.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Checkpoint directory internals, offset write-ahead logs (WAL), state recovery, and the conditions required for end-to-end exactly-once processing.
- **Why does this topic exist?**: Checkpoints allow streaming queries to restart after crashes from the exact point of interruption without data loss or duplication.
- **What problem does it solve?**: Stream restarting from scratch and duplicating side-effects on restart.
- **Where is it used in real-world Data Engineering?**: Zero-data-loss streaming pipelines.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Streaming queries, sources/sinks, delivery semantics |
| **Required depth** | Deep working knowledge: can design recoverable streaming pipelines and explain exactly-once conditions |
| **Data Engineer** | Configure checkpoints and reason about recovery and duplicate risk. |
| **Senior Data Engineer** | Design idempotent sinks and diagnose checkpoint corruption. |
| **Spark specialist** | Reason about offset commits, state recovery, and end-to-end guarantees. |
| **Next topic** | `09_streaming/06_kafka_integration.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A checkpoint records offsets, commits, and state so a streaming query can resume after failure. But a
checkpoint alone does not guarantee exactly-once: the source must be replayable, and the sink must tolerate
reprocessing through idempotent writes or transactions. The checkpoint is the bookkeeping; the source and sink
provide the guarantee.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Checkpoint** | Progress and state directory | Offsets, commits, metadata, state |
| **Offset** | Source position | Where the query left off |
| **Commit** | Batch completion record | Used to avoid reprocessing committed batches |
| **State store** | Aggregation/session state | Recovered from the checkpoint |
| **Idempotent sink** | Safe reprocessing | Same input produces the same output |
| **Transactional sink** | Atomic output | Commit aligns with checkpoint progress |

### How it works, step by step

1. The query reads a batch and computes the result.
2. It writes to the sink.
3. It records the batch commit and offsets in the checkpoint.
4. On restart, it reads the checkpoint and resumes from the last committed offset.
5. If the sink write succeeded but the commit did not, the batch may be reprocessed.
6. Idempotent or transactional sinks make that reprocessing safe.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Checkpoint Directory Structure: `offsets/`, `commits/`, `state/`, `metadata`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("Streaming_Checkpoint_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
chk_path = "/opt/spark-data/warehouse/chk_structure_demo"
os.system(f"rm -rf {chk_path}")

# Run short rate stream to generate checkpoint files
rate = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
q = rate.writeStream.format("noop").option("checkpointLocation", chk_path).start()
time.sleep(3)
q.stop()

print("Files created inside checkpoint directory:")
for root, dirs, files in os.walk(chk_path):
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), chk_path)
        print(f"  - {rel}")

---

## Experiment: Resuming Stream from Checkpoint

- **1. What are we testing?**: Restart stream pointing to same checkpoint and verify it resumes from saved offset.
- **2. What do we expect?**: Stream continues from last committed micro-batch without reprocessing batch 0.

In [ ]:
q2 = rate.writeStream.format("noop").option("checkpointLocation", chk_path).start()
time.sleep(2)
last_id = q2.lastProgress["batchId"] if q2.lastProgress else 0
print(f"Resumed stream active! Last processed batch ID: {last_id}")
q2.stop()

**3. What happened?**: Stream resumed without re-reading historical batches.  
**4. Why did it happen?**: The commit log confirmed previous batches were already acknowledged.  
**5. Practical Engineering Lesson**: Never delete the checkpoint directory if you need uninterrupted recovery.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Exactly-once ingestion into a lake

- **Industry**: Retail
- **Scenario**: A streaming job writes events to a partitioned table and must not duplicate rows after failures.
- **Problem**: A crash between the sink write and the checkpoint commit can replay a batch.
- **Why this concept is relevant**: A transactional table format or an idempotent merge keyed by event ID makes replay safe.
- **Implementation idea**: Use a table format with transactional commits or deduplicate by a deterministic event key.
- **Expected result**: No duplicates after repeated failures.
- **Alternative**: Accept at-least-once and deduplicate downstream.
- **Trade-offs**: Downstream dedup is simpler but shifts cost and may miss real-time consumers.

### Use Case 2: Checkpoint corruption after a configuration change

- **Industry**: Telecom
- **Scenario**: A job is restarted with a different query and fails to recover from the old checkpoint.
- **Problem**: The checkpoint encodes the query plan and state schema; incompatible changes cannot be resumed.
- **Why this concept is relevant**: A new query requires a new checkpoint or a planned state migration.
- **Implementation idea**: Version checkpoints with query releases and start a new checkpoint for breaking changes.
- **Expected result**: A recoverable deployment with a documented cutover.
- **Alternative**: Delete the checkpoint and reprocess from the beginning.
- **Trade-offs**: Reprocessing is simple but can be expensive and may violate downstream SLAs.

---

## Business Value

- Reliable recovery without duplicate business records.
- Clear ownership of checkpoint lifecycle.
- Safer deployments and schema changes.
- Auditable delivery semantics.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use a unique checkpoint location per query.
- Never share a checkpoint between queries.
- Make sinks idempotent or transactional.
- Version checkpoints with query changes.
- Monitor checkpoint size and commit latency.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Simulate a failure between sink and commit

**Steps**

1. Run a stream to an idempotent sink with a checkpoint.
2. Stop the query abruptly and restart it.
3. Verify that no duplicate business keys appear.

**What to observe**: A replay may occur, but the idempotent sink keeps the result correct.

**Healthy result**: Business key counts remain stable across restarts.

**Problematic result**: Duplicates appear because the sink is append-only.

**Engineering takeaway**: End-to-end correctness requires sink design, not just a checkpoint.

---

## Trade-offs and Alternatives

### Advantages

- Automatic recovery
- State restoration
- Progress tracking

### Limitations

- Not a guarantee by itself
- Query changes can invalidate it
- Checkpoint size grows with state

### When to use it

- Long-running streams
- Stateful aggregations
- Exactly-once designs

### When not to use it

- Sharing checkpoints
- Assuming exactly-once without sink support
- Deleting checkpoints casually

### Alternatives inside Spark

- Checkpoint
- State store
- Offset log

### Alternatives outside Spark

- Transactional table formats
- Kafka transactions
- Idempotent databases

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Commit Log
**Problem Statement**: Read the contents of the latest commit file in checkpoint.  
**Input / Context**: commits

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Exactly-Once Requirements
**Problem Statement**: List the 4 requirements for exactly-once streaming.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Changing Query Logic
**Problem Statement**: Can you change DataFrame query logic with an existing checkpoint?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Checkpoint incompatibility

**Scenario**: A deployment adds an aggregation and the job fails to restart from the existing checkpoint.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the error for state schema or plan incompatibility.
2. Decide whether a new checkpoint and backfill are acceptable.
3. Plan state migration if the change must preserve continuity.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Commit Log):
```python
print('Commit file records batch completion status.')
```

#### Solution 2 (Exactly-Once Requirements):
```python
print('1. Replayable source, 2. Deterministic execution, 3. Idempotent sink, 4. Checkpointing.')
```

#### Solution 3 (Changing Query Logic):
```python
print('Certain breaking schema changes or stateful operator mutations require starting a new checkpoint directory.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Exactly-Once**: Achieved through offset log + idempotent sink.
- **Checkpoint Integrity**: Must be stored on reliable storage.

### Essential Technical Terms
- `WAL`: Write-ahead log recording offsets.
- `Commit Log`: Confirms batch completion.

### Common Engineering Mistakes
- **Storing checkpoints on local /tmp**: Container restart wipes checkpoints, causing stream to reprocess from beginning.

### Interview Questions on This Topic
- **Q: What are the components of a streaming checkpoint directory?**
  - *A*: offsets/ (uncommitted source offsets), commits/ (completed batches), state/ (state store snapshots), and metadata (unique query run ID).


### More Interview Questions

- **Q: Does a checkpoint guarantee exactly-once?**
  - *A*: No. It records progress and state, but exactly-once also requires a replayable source and an idempotent or transactional sink.
- **Q: Why can a batch be reprocessed after a failure?**
  - *A*: If the sink write completed but the checkpoint commit did not, the query resumes from the previous committed offset and replays the batch.

### What to Study Next
Open 06_kafka_integration.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')